## IMPORT PACKAGES

In [31]:
import numpy as np
import pandas as pd
import cloudpickle
from pathlib import Path

# Autocomplete settings
%config IPCompleter.greedy=True

from janitor import clean_names
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

# project_path = Path('/Users/gloriadelriomarquez/Documents/Career/Projects/ml-lab/client_churn')
# sys.path.append(str(project_path))

from kkbox_churn.config import (PROJECT_ROOT, 
                        ID_COL, TARGET, DATETIME_COLS, NOMINAL_NUM_COLS,
                        NOMINAL_STR_COLS, INT_COLS, FLOAT_COLS,
                        BINARY_COLS, ZERO_COLS, MEDIAN_COLS, 
                        AGE_MIN, AGE_MAX, 
                        SELECTED_FEATURES, 
                        OHE_COLS, PASSTHROUGH_COLS, RS_COLS, RS_QUANTILE_RANGE )

## LOAD DATA

In [32]:
project_path = Path('/Users/gloriadelriomarquez/Documents/Career/Projects/ml-lab/client_churn')
df = pd.read_csv(project_path / "02_data" / "train" / "train.csv", index_col=0)

## DATASET STRUCTURE

In [33]:
### Correcting column names
df = clean_names(df)
df = df.rename(columns = {'bd':'age'})

### Prepare target and predictors

In [34]:
target = 'is_churn'

X = df[SELECTED_FEATURES].copy()   # predictors

y = df[target].copy()

## PIPELINE CREATION

In [35]:
### Instantiate data cleaning function
def cast_types(df):
    """
    Asigna los tipos de datos a cada grupo de variables.
    Devuelve una copia del DataFrame.
    """
    df = df.copy()

    # Comprobaciones previas
    all_cols = (DATETIME_COLS + NOMINAL_NUM_COLS + NOMINAL_STR_COLS
                + INT_COLS + FLOAT_COLS)

    duplicated = {c for c in all_cols if all_cols.count(c) > 1}
    assert not duplicated, f"Columnas en más de un grupo: {duplicated}"

    missing = set(all_cols) - set(df.columns)
    assert not missing, f"Columnas que no existen en el df: {missing}"

    remaining = set(df.columns) - set(all_cols) - {ID_COL, TARGET}
    assert not remaining, f"Columnas sin tipar: {remaining}"

    # Conversión de tipos
    df[DATETIME_COLS] = df[DATETIME_COLS].apply(pd.to_datetime)
    df[NOMINAL_NUM_COLS] = df[NOMINAL_NUM_COLS].astype("Int32").astype("string")
    df[NOMINAL_STR_COLS] = df[NOMINAL_STR_COLS].astype("string")
    df[INT_COLS] = df[INT_COLS].astype("Int32")
    df[FLOAT_COLS] = df[FLOAT_COLS].astype("float64")

    # El target solo existe en train y validación
    if TARGET in df.columns:
        df[TARGET] = df[TARGET].astype("Int8")

    return df


In [36]:
# AUXILIARY FUNCTIONS
def prepare_age(df):
    """Marca edades ausentes o fuera de rango y las convierte en NaN."""
    df = df.copy()
    valid_age = df["age"].between(AGE_MIN, AGE_MAX).fillna(False).astype(bool)
    # Variable que recoge la ausencia de info
    df["age_unknown"] = (~valid_age).astype("int8")
    df["age"] = df["age"].where(valid_age)
    return df


def fit_medians(df):
    """Calcula las medianas en train. Se ejecuta solo una vez."""
    df = prepare_age(cast_types(df))
    return {
        col: round(df[col].median()) if pd.api.types.is_integer_dtype(df[col])
        else df[col].median()
        for col in MEDIAN_COLS
    }

In [37]:
def clean_data(df, medians):
    """Limpia e imputa usando las medianas aprendidas en train."""
    temp = cast_types(df)
    temp = prepare_age(temp)

    # Nominales
    nominal_cols = NOMINAL_NUM_COLS + NOMINAL_STR_COLS
    temp[nominal_cols] = temp[nominal_cols].fillna("Unknown")

    # Binarias (2 = desconocido)
    temp[BINARY_COLS] = temp[BINARY_COLS].fillna(2)

    # Numéricas
    temp[ZERO_COLS] = temp[ZERO_COLS].fillna(0)
    temp[MEDIAN_COLS] = temp[MEDIAN_COLS].fillna(medians)

    # Comprobación final
    imputed_cols = nominal_cols + BINARY_COLS + ZERO_COLS + MEDIAN_COLS
    remaining_na = temp[imputed_cols].isna().sum()
    assert remaining_na.sum() == 0, f"Quedan NaN: {remaining_na[remaining_na > 0].to_dict()}"

    return temp

####### USO ########
# medians = fit_medians(train)
# train_clean = clean_data(train, medians)
# val_clean = clean_data(val, medians)

### FunctionTransformer

In [38]:
# Wrap data cleaning as a custom transformer
# Medians are learned in fit() (train only) and reused in transform()

from sklearn.base import BaseEstimator, TransformerMixin

class DataQualityTransformer(BaseEstimator, TransformerMixin):
    """Tipado, limpieza e imputación con medianas aprendidas en fit."""

    def fit(self, X, y=None):
        self.medians_ = fit_medians(X)
        return self

    def transform(self, X):
        return clean_data(X, self.medians_)

data_quality_transformer = DataQualityTransformer()

### Column transformer

In [39]:
# Instantiate variable transformations
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler


ohe = OneHotEncoder(drop=None, sparse_output=False, handle_unknown="ignore")
rs = RobustScaler(quantile_range=RS_QUANTILE_RANGE)

preprocessor = ColumnTransformer(
    transformers=[
        ("ohe", ohe, OHE_COLS),
        ("rs", rs, RS_COLS),
        ("bin", "passthrough", PASSTHROUGH_COLS),
    ],
    remainder="drop"    # descarta msno, fechas y target
)

### Pipelines

In [40]:
# Data preparation pipeline
preprocessing_pipeline = make_pipeline(data_quality_transformer, preprocessor) 

# Instantiate model
model = HistGradientBoostingClassifier(early_stopping = True,
                                        validation_fraction = 0.1,
                                        n_iter_no_change = 20,
                                        max_iter = 1000,
                                        learning_rate = 0.1,
                                        max_leaf_nodes = 63,
                                        min_samples_leaf = 200,
                                        l2_regularization = 0,
                                        random_state = 42
                                        )

# Training pipeline
training_pipeline = make_pipeline(preprocessing_pipeline,model)

## SAVE TRAINING PIPELINE

In [41]:
# Save the training pipeline
training_pipeline_name = 'training_pipeline.pickle'
training_pipeline_path = project_path / '04_models' / training_pipeline_name

with open(training_pipeline_path, mode='wb') as file:
    cloudpickle.dump(training_pipeline, file)

## SAVE EXECUTION PIPELINE

In [42]:
# Train the final execution pipeline
execution_pipeline = training_pipeline.fit(X, y)
# Execution pipeline filename
execution_pipeline_name = 'execution_pipeline.pickle'

# Save the final execution pipeline
execution_pipeline_path = PROJECT_ROOT / '04_models' / execution_pipeline_name

with open(execution_pipeline_path, mode='wb') as file:
    cloudpickle.dump(execution_pipeline, file)